In [1]:
# @title 导入库


import dataclasses
import datetime
import functools
import math
import re
from typing import Optional

import cartopy.crs as ccrs
#from google.cloud import storage
from graphcast import autoregressive
from graphcast import casting
from graphcast import checkpoint
from graphcast import data_utils
from graphcast import graphcast
from graphcast import normalization
from graphcast import rollout
from graphcast import xarray_jax
from graphcast import xarray_tree
from IPython.display import HTML
import ipywidgets as widgets
import haiku as hk
import jax
import matplotlib
import matplotlib.pyplot as plt
from matplotlib import animation
import numpy as np
import xarray

import os
os.environ["JAX_DISABLE_XLA"] = "1"



def parse_file_parts(file_name):
  return dict(part.split("-", 1) for part in file_name.split("_"))


In [2]:
# @title 载入绘图函数

# 这定义了一个名为 select 的函数，它接受一个 xarray.Dataset 对象、一个指定数据集中变量的字符串，以及可选的水平和最大时间步数参数。它返回一个 xarray.Dataset。
def select(
    data: xarray.Dataset,
    variable: str,
    level: Optional[int] = None,
    max_steps: Optional[int] = None
    ) -> xarray.Dataset:
#     从数据集中选择与指定变量相对应的数据。
  data = data[variable]
#     如果数据集有一个名为 “batch” 的维度，这行代码选择数据的第一批。
  if "batch" in data.dims:
    data = data.isel(batch=0)
#     如果指定了 max_steps，并且数据集有一个 “time” 维度且步数多于 max_steps，这行代码将数据集限制在前 max_steps 个时间步。
  if max_steps is not None and "time" in data.sizes and max_steps < data.sizes["time"]:
    data = data.isel(time=range(0, max_steps))
#     如果指定了 level 并且它存在于数据集的坐标中，这行代码选择在该特定水平上的数据。
  if level is not None and "level" in data.coords:
    data = data.sel(level=level, method="nearest")
  return data

# 这定义了一个名为 scale 的函数，它接受一个 xarray.Dataset、一个用于缩放的可选中心值和一个表示是否使用鲁棒缩放的布尔值。
# 它返回一个包含数据集、matplotlib.colors.Normalize 对象和颜色映射名称的元组。
def scale(
    data: xarray.Dataset,
    center: Optional[float] = None,
    robust: bool = False,
    ) -> tuple[xarray.Dataset, matplotlib.colors.Normalize, str]:
#     这些行计算用于归一化的最小和最大值。如果 robust 为 True，它使用第2和第98百分位数来忽略异常值；否则，它使用绝对最小和最大值。
  vmin = np.nanpercentile(data, (2 if robust else 0))
  vmax = np.nanpercentile(data, (98 if robust else 100))
# 如果提供了 center 值，这些行调整 vmin 和 vmax 使其与中心等距，确保中心值是颜色刻度的中点。
  if center is not None:
    diff = max(vmax - center, center - vmin)
    vmin = center - diff
    vmax = center + diff
#     函数返回数据集、配置有 vmin 和 vmax 的 Normalize 对象，以及要使用的颜色映射名称（如果有中心点则为 “RdBu_r”，否则为 “viridis”）。
  return (data, matplotlib.colors.Normalize(vmin, vmax),
          ("RdBu_r" if center is not None else "viridis"))

# 这定义了一个名为 plot_data 的函数，它接受一个标题和 xarray.Dataset 对象的字典、图形标题、可选的绘图大小、鲁棒缩放标志和子图布局的列数。
# 它返回一个类似于 scale 函数的元组。
def plot_data(
    data: dict[str, xarray.Dataset],
    fig_title: str,
    plot_size: float = 5,
    robust: bool = False,
    cols: int = 4
    ) -> tuple[xarray.Dataset, matplotlib.colors.Normalize, str]:

# 这些行获取字典中的第一个数据集以确定时间步数（max_steps），并断言所有数据集都有相同数量的时间步。
  first_data = next(iter(data.values()))[0]
  max_steps = first_data.sizes.get("time", 1)
  assert all(max_steps == d.sizes.get("time", 1) for d, _, _ in data.values())

#     列数设置为指定列数或数据字典长度的最小值。根据列数计算行数。
  cols = min(cols, len(data))
  rows = math.ceil(len(data) / cols)
# 创建一个新的图形，其大小基于行数和列数以及指定的绘图大小。
  figure = plt.figure(figsize=(plot_size * 2 * cols,
                               plot_size * rows))
#     设置图形的标题，并调整布局以消除子图之间的空白。
  figure.suptitle(fig_title, fontsize=16)
  figure.subplots_adjust(wspace=0, hspace=0)
  figure.tight_layout()

#     这个循环为字典中的每个数据集创建子图，设置轴和标题。
  images = []
  for i, (title, (plot_data, norm, cmap)) in enumerate(data.items()):
    ax = figure.add_subplot(rows, cols, i+1)
    ax.set_xticks([])
    ax.set_yticks([])
    ax.set_title(title)
#     每个子图使用 imshow 函数显示数据集的第一个时间步，使用指定的归一化和颜色映射。
    im = ax.imshow(
        plot_data.isel(time=0, missing_dims="ignore"), norm=norm,
        origin="lower", cmap=cmap)
#     为每个子图添加一个颜色条。
    plt.colorbar(
        mappable=im,
        ax=ax,
        orientation="vertical",
        pad=0.02,
        aspect=16,
        shrink=0.75,
        cmap=cmap,
        extend=("both" if robust else "neither"))
    images.append(im)

#     这定义了一个用于动画的 update 函数，它会在每个帧更新标题和数据。
  def update(frame):
    if "time" in first_data.dims:
      td = datetime.timedelta(microseconds=first_data["time"][frame].item() / 1000)
      figure.suptitle(f"{fig_title}, {td}", fontsize=16)
    else:
      figure.suptitle(fig_title, fontsize=16)
    for im, (plot_data, norm, cmap) in zip(images, data.values()):
      im.set_data(plot_data.isel(time=frame, missing_dims="ignore"))

#     使用 FuncAnimation 类创建一个动画，它将为每个帧调用 update 函数。
  ani = animation.FuncAnimation(
      fig=figure, func=update, frames=max_steps, interval=250)
#     关闭图形（以防止它立即显示），并将动画转换为使用 JavaScript 的 HTML5 视频，然后返回。
  plt.close(figure.number)
  return HTML(ani.to_jshtml())

In [3]:
# @title 选择模型
# Rewrite by S.F. Sune, https://github.com/sfsun67.
'''
    我们有三种训练好的模型可供选择, 需要从https://console.cloud.google.com/storage/browser/dm_graphcast准备：
    GraphCast - ERA5 1979-2017 - resolution 0.25 - pressure levels 37 - mesh 2to6 - precipitation input and output.npz
    GraphCast_operational - ERA5-HRES 1979-2021 - resolution 0.25 - pressure levels 13 - mesh 2to6 - precipitation output only.npz
    GraphCast_small - ERA5 1979-2015 - resolution 1.0 - pressure levels 13 - mesh 2to5 - precipitation input and output.npz
'''
# 在此路径 /root/data/params 中查找结果，并列出 "params/"中所有文件的名称，去掉名称中的 "params/"perfix。

import os
import glob

# 定义数据目录，请替换成你自己的目录。
dir_path_params = "/root/code/GraphCast-from-Ground-Zero"


# Use glob to get all file paths in the directory
# 使用glob.glob函数和os.path.join方法获取dir_path_params目录下所有文件的路径。
file_paths_params = glob.glob(os.path.join(dir_path_params, "*"))

# 使用glob.glob函数和os.path.join方法获取dir_path_params目录下所有文件的路径。
# Remove the directory path and the ".../params/" prefix from each file name
params_file_options = [os.path.basename(path) for path in file_paths_params]

# 创建一个整数滑块控件random_mesh_size，用于选择网格大小。其默认值为4，最小值为4，最大值为6。较大的网格可以捕获更细的空间特征，但也会增加计算成本。
random_mesh_size = widgets.IntSlider(
    value=6, min=4, max=6, description="Mesh size:")
# 创建一个整数滑块控件random_gnn_msg_steps，用于选择GNN消息传递的步数。其默认值为4，最小值为1，最大值为32。较多的步数能让每个节点接收到来自更远邻居的信息，但会增加计算量和训练时间。
random_gnn_msg_steps = widgets.IntSlider(
    value=8, min=1, max=64, description="GNN message steps:")
# 创建一个下拉菜单控件random_latent_size，用于选择潜在大小。选项是2的4次方到2的9次方，即16到512，其默认值为32。较大的 latent_size 能够捕获更多的信息，但也可能导致过拟合或增加计算成本。
random_latent_size = widgets.Dropdown(
    options=[int(2**i) for i in range(4, 10)], value=16,description="Latent size:")
# 创建一个下拉菜单控件random_levels，用于选择压力水平。选项有13和37，其默认值为13。
random_levels = widgets.Dropdown(
    options=[3, 37], value=3, description="Pressure levels:")

# 创建一个下拉菜单控件params_file，用于选择参数文件。选项是之前从目录中获取的文件名列表。
params_file = widgets.Dropdown(
    options=params_file_options,
    description="Params file:",
    layout={"width": "max-content"})

# 创建一个标签页控件source_tab，其中包含两个标签。第一个标签是一个垂直布局的盒子，包含了前面创建的四个控件。第二个标签是参数文件的下拉菜单。
source_tab = widgets.Tab([
    widgets.VBox([
        random_mesh_size,
        random_gnn_msg_steps,
        random_latent_size,
        random_levels,
    ]),
    params_file,
])
# 为source_tab的两个标签设置标题，分别为"随机参数权重"和"预训练权重"。
source_tab.set_title(0, "随机参数权重（Random）")
source_tab.set_title(1, "预训练权重（Checkpoint）")
# 最后，创建一个垂直布局的盒子，包含了source_tab和一个标签，提示用户运行下一个单元格以加载模型，并告知重新运行该单元格将清除他们的选择。
widgets.VBox([
    source_tab,
    widgets.Label(value="运行下一个单元格以加载模型。重新运行该单元格将清除您的选择。")
])


In [4]:
# @title 加载模型

# 这行代码获取当前选中的标签页的标题，以确定用户选择了哪种参数权重（随机参数权重或预训练权重）。
source = source_tab.get_title(source_tab.selected_index)

# 如果用户选择了“随机参数权重”，则执行以下代码块。
if source == "随机参数权重（Random）":
#     初始化params为None和state为一个空字典。这些将在后面的代码中使用
  params = None  # Filled in below
  state = {}
# 创建一个model_config对象，它包含模型配置的参数。这些参数包括：
# resolution：分辨率，这里设置为0。
# mesh_size：网格大小，取自之前创建的滑块控件的值。
# latent_size：潜在大小，取自下拉菜单控件的值。
# gnn_msg_steps：GNN消息传递步数，取自滑块控件的值。
# hidden_layers：隐藏层的数量，这里设置为1。
# radius_query_fraction_edge_length：查询半径与边长的比例，这里设置为0.6。
  model_config = graphcast.ModelConfig(
      resolution=0,
      mesh_size=random_mesh_size.value,
      latent_size=random_latent_size.value,
      gnn_msg_steps=random_gnn_msg_steps.value,
      hidden_layers=1,
      radius_query_fraction_edge_length=0.6)
#     创建一个task_config对象，它包含任务配置的参数。这些参数包括：
# input_variables：输入变量。
# target_variables：目标变量。
# forcing_variables：强迫变量。
# pressure_levels：压力水平，取自下拉菜单控件的值。
# input_duration：输入持续时间。
  task_config = graphcast.TaskConfig(
      input_variables=graphcast.TASK.input_variables,
      target_variables=graphcast.TASK.target_variables,
      forcing_variables=graphcast.TASK.forcing_variables,
      pressure_levels=graphcast.PRESSURE_LEVELS[random_levels.value],
      input_duration=graphcast.TASK.input_duration,
  )
#     如果用户选择了“预训练权重”，则执行以下代码块。
# 这段被注释的代码原本用于从Google Cloud Storage加载预训练权重。
# 现在，它被替换为从本地文件系统加载预训练权重的代码。使用open函数以二进制读取模式打开参数文件，并使用checkpoint.load函数加载检查点。
else:
  assert source == "预训练权重（Checkpoint）"
  '''with gcs_bucket.blob(f"params/{params_file.value}").open("rb") as f:
    ckpt = checkpoint.load(f, graphcast.CheckPoint)'''
  
  with open(f"{dir_path_params}/{params_file.value}", "rb") as f:
    ckpt = checkpoint.load(f, graphcast.CheckPoint)
    
#  从检查点中提取参数到params变量，并重新初始化state为一个空字典。 
  params = ckpt.params
  state = {}

# 从检查点中提取模型配置和任务配置。
  model_config = ckpt.model_config
  task_config = ckpt.task_config
  print("模型描述:\n", ckpt.description, "\n")
  print("模型许可信息:\n", ckpt.license, "\n")

model_config

ModelConfig(resolution=0, mesh_size=6, latent_size=16, gnn_msg_steps=8, hidden_layers=1, radius_query_fraction_edge_length=0.6, mesh2grid_edge_normalization_factor=None, patch_size_lat=8, patch_size_lon=8)

In [5]:
# import xarray as xr 
# import pandas as pd
# import numpy as np

# dir_path_data = "/root/autodl-tmp/"
# batch_size = 12

# # 使用 dask 进行懒加载
# ds = xr.open_dataset(f"{dir_path_data}/cmems_mod_glo_phy_my_0.083deg_P1D-m_multi-vars_180.00W-179.92E_80.00S-90.00N_0.49-1.54m_2020-08-28-2020-09-28.nc", chunks={'time': batch_size})

# # 重命名维度
# ds = ds.rename({'latitude': 'lat', 'longitude': 'lon', 'depth': 'level'})

# # 计算新的批次维度大小
# num_batches = ds.dims['time'] // batch_size

# all_batches = []

# # 分批处理数据
# for i in range(num_batches):
#     start = i * batch_size
#     end = start + batch_size
#     batch_data = ds.isel(time=slice(start, end))
#     all_batches.append(batch_data)

# # 合并所有批次的数据集
# eval_batch = xr.concat(all_batches, dim='time')
# eval_batch = eval_batch.astype(np.float32)

# # 创建新的时间坐标数组
# time_coords = eval_batch.coords['time'].values
# datetime_coords = np.full((num_batches, batch_size), np.datetime64('NaT'), dtype='datetime64[ns]')

# for i in range(num_batches):
#     start = i * batch_size
#     end = (i + 1) * batch_size
#     datetime_coords[i, :] = time_coords[start:end]

# # 创建新的数据变量字典，重塑数据
# new_data_vars = {}
# for var in eval_batch.data_vars:
#     data = eval_batch[var].values
#     reshaped_data = data.reshape((num_batches, batch_size, *data.shape[1:]))
#     new_data_vars[var] = (['batch', 'time'] + list(eval_batch[var].dims[1:]), reshaped_data)

# # 创建新的数据集，添加 batch 和 time 维度
# eval_batch = xr.Dataset(new_data_vars, coords={ 
#     'level': eval_batch['level'],
#     'lon': eval_batch['lon'],
#     'lat': eval_batch['lat'],
#     'time': ('time', np.arange(batch_size)),  # 使用0到batch_size-1的整数作为新的时间坐标
#     'batch': ('batch', np.arange(num_batches)),
#     'datetime': (['batch', 'time'], datetime_coords),
# })

# # 恢复 time 维度为 timedelta64[ns]
# eval_batch['time'] = pd.to_timedelta(eval_batch['time'], unit='D')

# # 重排维度并移除坐标中的 batch
# eval_batch = eval_batch.drop_vars('batch')

# # # 检查并生成新的均匀间隔的坐标
# # new_lat = np.linspace(eval_batch.lat.min(), eval_batch.lat.max(), eval_batch.dims['lat'])
# # new_lon = np.linspace(eval_batch.lon.min(), eval_batch.lon.max(), eval_batch.dims['lon'])

# # # 对数据集进行重采样
# # eval_batch = eval_batch.interp(lat=new_lat, lon=new_lon)
# # # # 确保数据类型为 float32
# eval_batch = eval_batch.astype(np.float32)

# dir_path_data = "/root/autodl-tmp/"
# output_file_path = f"{dir_path_data}/eval_batch.nc"
# eval_batch.to_netcdf(output_file_path)

# # 显示修改后的数据集
# eval_batch

In [6]:
# # 检查间隔
# print("New latitude interval:", np.diff(eval_batch.lat))
# print("New longitude interval:", np.diff(eval_batch.lon))

In [7]:
# import xarray as xr
# import pandas as pd
# import numpy as np
# import gc  # 引入垃圾回收模块

# # 数据路径
# dir_path_data = "/root/autodl-tmp/"
# input_file_path = f"{dir_path_data}/eval_batchall.nc"

# eval_batch = xr.open_dataset(input_file_path, chunks={'lat': 100, 'lon': 100})

# # 将 time 转换为 timedelta64[ns]
# eval_batch['time'] = pd.to_timedelta(eval_batch['time'], unit='D')

# eval_batch = eval_batch.drop_vars('batch')

# # 生成新的均匀网格
# new_lat = np.linspace(eval_batch.lat.min(), eval_batch.lat.max(), eval_batch.dims['lat'])
# new_lon = np.linspace(eval_batch.lon.min(), eval_batch.lon.max(), eval_batch.dims['lon'])

# # 逐变量处理和保存
# for var_name, var_data in eval_batch.data_vars.items():
#     print(f"正在处理变量: {var_name}")

#     # 插值
#     interpolated_data = var_data.interp(lat=new_lat, lon=new_lon)

#     # 保存每个变量到单独的文件
#     output_file_path = f"{dir_path_data}/{var_name}_interp.nc"
#     print(f"保存插值结果到: {output_file_path}")
#     interpolated_data.to_netcdf(output_file_path)

#     # 释放内存
#     del interpolated_data
#     gc.collect()

# # 合并所有插值结果为一个数据集（可选）
# print("开始合并所有变量到一个文件...")
# output_file_path_all = f"{dir_path_data}/eval_batchall_interp.nc"
# eval_batch = xr.merge([
#     xr.open_dataset(f"{dir_path_data}/{var_name}_interp.nc")
#     for var_name in eval_batch.data_vars.keys()
# ])

# # 保存合并后的数据集
# eval_batch.to_netcdf(output_file_path_all)
# print(f"最终插值数据保存到: {output_file_path_all}")

# print("所有插值完成！")
# eval_batch

In [8]:
import xarray as xr 
import numpy as np

dir_path_data = "/root/autodl-tmp/"
# 读取 NetCDF 文件到 eval_batch
input_file_path = f"{dir_path_data}/eval_batch.nc"
eval_batch = xr.open_dataset(input_file_path, chunks={'batch': 1})

# 确保数据类型为 float32
eval_batch = eval_batch.astype(np.float32)
# eval_batch = evacl_batch.slice(batch=)

# 如果需要立即计算所有数据，可以使用 .load() 或 .compute() 
# eval_batch = eval_batch.load()  # 或者 eval_batch = eval_batch.compute()

# 输出降采样后的数据集
eval_batch

<xarray.Dataset>
Dimensions:   (batch: 1, time: 12, lat: 2041, lon: 4320, level: 3)
Coordinates:
  * level     (level) float32 0.494 2.646 5.078
  * lon       (lon) float32 -180.0 -179.9 -179.8 -179.8 ... 179.8 179.8 179.9
  * lat       (lat) float32 -80.0 -79.92 -79.83 -79.75 ... 89.83 89.92 90.0
  * time      (time) timedelta64[ns] 0 days 1 days 2 days ... 10 days 11 days
    datetime  (batch, time) datetime64[ns] dask.array<chunksize=(1, 12), meta=np.ndarray>
Dimensions without coordinates: batch
Data variables:
    siconc    (batch, time, lat, lon) float32 dask.array<chunksize=(1, 12, 2041, 4320), meta=np.ndarray>
    sithick   (batch, time, lat, lon) float32 dask.array<chunksize=(1, 12, 2041, 4320), meta=np.ndarray>
    so        (batch, time, level, lat, lon) float32 dask.array<chunksize=(1, 12, 3, 2041, 4320), meta=np.ndarray>
    thetao    (batch, time, level, lat, lon) float32 dask.array<chunksize=(1, 12, 3, 2041, 4320), meta=np.ndarray>
    uo        (batch, time, level, lat, lon) float32 dask.array<chunksize=(1, 12, 3, 2041, 4320), meta=np.ndarray>
    usi       (batch, time, lat, lon) float32 dask.array<chunksize=(1, 12, 2041, 4320), meta=np.ndarray>
    vo        (batch, time, level, lat, lon) float32 dask.array<chunksize=(1, 12, 3, 2041, 4320), meta=np.ndarray>
    vsi       (batch, time, lat, lon) float32 dask.array<chunksize=(1, 12, 2041, 4320), meta=np.ndarray>
    zos       (batch, time, lat, lon) float32 dask.array<chunksize=(1, 12, 2041, 4320), meta=np.ndarray>

In [9]:
# # 创建一个新的变量 land_sea_mask
# import xarray as xr
# import numpy as np

# # 延迟加载数据以降低内存占用（如果数据规模较大）
# eval_batch = eval_batch.chunk({"lat": 100, "lon": 100})

# # 创建 land_sea_mask 变量，避免中间副本
# eval_batch["land_sea_mask"] = (
#     (~np.isnan(eval_batch["so"].isel(time=0, level=0)))
#     .astype(np.float32)
#     .persist()  # 在 Dask 中持久化计算结果以减少重复计算
# )

# eval_batch = eval_batch.fillna(0)
# eval_batch = eval_batch.astype(np.float32)
# # eval_batch = eval_batch.isel(time=slice(0, 7))


In [10]:
# dir_path_data = "/root/autodl-tmp/"
# output_file_path = f"{dir_path_data}/eval_batch.nc"
# eval_batch.to_netcdf(output_file_path)

In [11]:
# # @title Choose data to plot

# plot_example_variable = widgets.Dropdown(
#     options=eval_batch.data_vars.keys(),
#     value="so",
#     description="Variable")
# plot_example_level = widgets.Dropdown(
#     options=eval_batch.coords["level"].values,
#     value=0.494025,
#     description="Level")
# plot_example_robust = widgets.Checkbox(value=True, description="Robust")
# plot_example_max_steps = widgets.IntSlider(
#     min=1, max=eval_batch.dims["time"], value=eval_batch.dims["time"],
#     description="Max steps")

# widgets.VBox([
#     plot_example_variable,
#     plot_example_level,
#     plot_example_robust,
#     plot_example_max_steps,
#     widgets.Label(value="Run the next cell to plot the data. Rerunning this cell clears your selection.")
# ])

In [12]:
# # @title Plot example data

# plot_size = 7

# data = {
#     " ": scale(select(eval_batch, plot_example_variable.value, plot_example_level.value, plot_example_max_steps.value),
#               robust=plot_example_robust.value),
# }
# fig_title = plot_example_variable.value
# if "level" in example_batch[plot_example_variable.value].coords:
#   fig_title += f" at {plot_example_level.value} hPa"

# plot_data(data, fig_title, plot_size, plot_example_robust.value)


In [13]:
# # 创建一个新的变量 land_sea_mask
# import xarray as xr
# import numpy as np

# eval_batch land_sea_mask 变量，避免中间副本
eval_batch["land_sea_mask"] = (
    (~np.isnan(eval_batch["so"].isel(time=0, level=0)))
    .astype(np.float32)
)

eval_batch = eval_batch.fillna(0)
eval_batch = eval_batch.astype(np.float32)
# example_batch

In [14]:
import numpy as np

def load_params_from_npz(filepath):
    # 加载 .npz 文件，启用 allow_pickle
    loaded = np.load(filepath, allow_pickle=True)
    
    # 将加载的内容转换为字典
    params = {key: loaded[key].item() if loaded[key].dtype == object else loaded[key] for key in loaded.files}
    
    print(f"模型参数已从 {filepath} 加载")
    return params

# 定义文件路径
load_filepath = "model_params_all_2.npz"

# 调用加载函数，读取参数
params = load_params_from_npz(load_filepath)

模型参数已从 model_params_all_2.npz 加载


In [15]:
# @title 加载规范化数据
# Rewrite by S.F. Sune, https://github.com/sfsun67.
# 定义了一个变量dir_path_stats，它存储了数据统计文件所在的目录路径。
import xarray
import os

dir_path_stats = "/root/data/stats/"

# 这行代码使用with语句和open函数以二进制读取模式（“rb”）打开一个名为stats-diffs_stddev_by_level.nc的文件。
# with open(f"{dir_path_stats}/stats-diffs_stddev_by_level.nc", "rb") as f:
#   diffs_stddev_by_level = xarray.load_dataset(f).compute()
# 类似于第4行，这行代码打开了另一个文件stats-mean_by_level.nc。
with open(f"{dir_path_stats}/stats-mean_by_level.nc", "rb") as f:
  mean_by_level = xarray.load_dataset(f).compute()
# 再次类似于第4行，这行代码打开了第三个文件stats-stddev_by_level.nc。
with open(f"{dir_path_stats}/stats-stddev_by_level.nc", "rb") as f:
  stddev_by_level = xarray.load_dataset(f).compute()
# 这行代码使用with语句和open函数以二进制读取模式（“rb”）打开一个名为stats-diffs_stddev_by_level.nc的文件。
with open(f"{dir_path_stats}/stats-diffs_stddev_by_level.nc", "rb") as f:
  diffs_stddev_by_level = xarray.load_dataset(f).compute()

In [16]:
test_batch = xr.Dataset({var: eval_batch[var].isel(batch=slice(0, 1)) for var in eval_batch.data_vars})
test_batch.load()

print(test_batch.dims.mapping)

# # 检查并调整 batch 维度
# if 'batch' not in train_batch.dims:
#     for var in train_batch.data_vars:
#         train_batch[var] = train_batch[var].expand_dims(batch=1)
# print(train_batch.dims.mapping)

test_batch

{'lon': 4320, 'lat': 2041, 'time': 12, 'batch': 1, 'level': 3}


<xarray.Dataset>
Dimensions:        (lon: 4320, lat: 2041, time: 12, batch: 1, level: 3)
Coordinates:
  * lon            (lon) float32 -180.0 -179.9 -179.8 ... 179.8 179.8 179.9
  * lat            (lat) float32 -80.0 -79.92 -79.83 -79.75 ... 89.83 89.92 90.0
  * time           (time) timedelta64[ns] 0 days 1 days ... 10 days 11 days
    datetime       (batch, time) datetime64[ns] 2019-01-01 ... 2019-01-12
  * level          (level) float32 0.494 2.646 5.078
Dimensions without coordinates: batch
Data variables:
    siconc         (batch, time, lat, lon) float32 0.0 0.0 0.0 ... 0.0 0.0 0.0
    sithick        (batch, time, lat, lon) float32 0.0 0.0 0.0 ... 0.0 0.0 0.0
    so             (batch, time, level, lat, lon) float32 0.0 0.0 ... 0.0 0.0
    thetao         (batch, time, level, lat, lon) float32 0.0 0.0 ... 0.0 0.0
    uo             (batch, time, level, lat, lon) float32 0.0 0.0 ... 0.0 0.0
    usi            (batch, time, lat, lon) float32 0.0 0.0 0.0 ... 0.0 0.0 0.0
    vo             (batch, time, level, lat, lon) float32 0.0 0.0 ... 0.0 0.0
    vsi            (batch, time, lat, lon) float32 0.0 0.0 0.0 ... 0.0 0.0 0.0
    zos            (batch, time, lat, lon) float32 0.0 0.0 0.0 ... 0.0 0.0 0.0
    land_sea_mask  (batch, lat, lon) float32 0.0 0.0 0.0 0.0 ... 0.0 0.0 0.0 0.0

In [17]:
# @title 选择要提取的训练和评估数据
eval_steps = widgets.IntSlider(
    value=1, min=1, max=test_batch.sizes["time"]-2, description="评估步数")

widgets.VBox([
    eval_steps,
    widgets.Label(value="运行下一个单元格以提取数据。重新运行此单元格将清除您的选择。")
])

In [18]:
# @title 提取训练和评估数据

print("数据集的维度:", eval_batch.dims)

# 打印数据集的坐标
print("数据集的坐标:", eval_batch.coords)

# 调用了一个名为 data_utils.extract_inputs_targets_forcings 的函数，并将其返回的结果分配给三个变量：train_inputs、train_targets 和 train_forcings。
eval_inputs, eval_targets, eval_forcings = data_utils.extract_inputs_targets_forcings(
#     example_batch 是一个示例数据批次，target_lead_times 是一个时间切片，用于选择目标数据的时间范围。在这里，我们选择了从24小时到训练步数乘以24小时的时间范围。
    test_batch, target_lead_times=slice("24h", f"{eval_steps.value*24}h"),
#     这是函数的第二个参数，它使用 dataclasses.asdict 将 task_config 转换为字典，并将其作为关键字参数传递给函数。
    **dataclasses.asdict(task_config))

print("所有示例：  ", eval_batch.dims.mapping)
print("训练输入：  ", eval_inputs.dims.mapping)
print("训练目标： ", eval_targets.dims.mapping)
print("训练强迫：", eval_forcings.dims.mapping)

数据集的维度: Frozen({'batch': 1, 'time': 12, 'lat': 2041, 'lon': 4320, 'level': 3})
数据集的坐标: Coordinates:
  * level     (level) float32 0.494 2.646 5.078
  * lon       (lon) float32 -180.0 -179.9 -179.8 -179.8 ... 179.8 179.8 179.9
  * lat       (lat) float32 -80.0 -79.92 -79.83 -79.75 ... 89.83 89.92 90.0
  * time      (time) timedelta64[ns] 0 days 1 days 2 days ... 10 days 11 days
    datetime  (batch, time) datetime64[ns] dask.array<chunksize=(1, 12), meta=np.ndarray>
所有示例：   {'batch': 1, 'time': 12, 'lat': 2041, 'lon': 4320, 'level': 3}
训练输入：   {'batch': 1, 'time': 2, 'level': 3, 'lat': 2041, 'lon': 4320}
训练目标：  {'batch': 1, 'time': 1, 'level': 3, 'lat': 2041, 'lon': 4320}
训练强迫： {'batch': 1, 'time': 1, 'lat': 2041, 'lon': 4320}


In [19]:
# 定义构建和包装GraphCast预测器的函数
def construct_wrapped_graphcast(
    model_config: graphcast.ModelConfig,
    task_config: graphcast.TaskConfig):
  """Constructs and wraps the GraphCast Predictor."""
  # 创建一个更深层次的一步预测器
  predictor = graphcast.GraphCast(model_config, task_config)

  # 修改输入/输出以处理从float32到BFloat16的转换
  # predictor = casting.Bfloat16Cast(predictor)

  # 在应用输入/目标的规范化之后，进行BFloat16的转换
  predictor = normalization.InputsAndResiduals(
      predictor,
      diffs_stddev_by_level=diffs_stddev_by_level,
      mean_by_level=mean_by_level,
      stddev_by_level=stddev_by_level)

  # 包装所有内容，使一步模型能够产生轨迹
  predictor = autoregressive.Predictor(predictor, gradient_checkpointing=True)
  return predictor

# 定义前向运算函数
@hk.transform_with_state
def run_forward(model_config, task_config, inputs, targets_template, forcings):
  predictor = construct_wrapped_graphcast(model_config, task_config)
  return predictor(inputs, targets_template=targets_template, forcings=forcings)

# 定义计算损失函数的函数
@hk.transform_with_state
def loss_fn(model_config, task_config, inputs, targets, forcings):
  predictor = construct_wrapped_graphcast(model_config, task_config)
  loss, diagnostics = predictor.loss(inputs, targets, forcings)
  return xarray_tree.map_structure(
      lambda x: xarray_jax.unwrap_data(x.mean(), require_jax=True),
      (loss, diagnostics))

# 定义计算梯度的函数
def grads_fn(params, state, model_config, task_config, inputs, targets, forcings):
  def _aux(params, state, i, t, f):
    (loss, diagnostics), next_state = loss_fn.apply(
        params, state, jax.random.PRNGKey(0), model_config, task_config,
        i, t, f)
    return loss, (diagnostics, next_state)
  (loss, (diagnostics, next_state)), grads = jax.value_and_grad(
      _aux, has_aux=True)(params, state, inputs, targets, forcings)
  return loss, diagnostics, next_state, grads

# 定义一个函数，用于通过functools.partial传递配置
def with_configs(fn):
  return functools.partial(
      fn, model_config=model_config, task_config=task_config)

# 定义一个函数，用于通过functools.partial传递参数和状态
def with_params(fn):
  return functools.partial(fn, params=params, state=state)

# 定义一个函数，用于丢弃状态并只返回预测结果
def drop_state(fn):
  return lambda **kw: fn(**kw)[0]

# 使用jax.jit编译初始化函数
init_jitted = jax.jit(with_configs(run_forward.init))

# 编译损失函数和梯度函数
loss_fn_jitted = drop_state(with_params(jax.jit(with_configs(loss_fn.apply))))
grads_fn_jitted = with_params(jax.jit(with_configs(grads_fn)))
run_forward_jitted = drop_state(with_params(jax.jit(with_configs(
    run_forward.apply))))


In [20]:
import xarray as xr
import numpy as np
import jax

def apply_mask(predictions, targets, mask):
    """应用掩码，只保留有效值。"""
    # 将 xarray.Dataset 中的 DataArray 转换为 numpy 数组
    masked_preds = np.where(mask, predictions, np.nan)  # 确保 predictions 是 DataArray
    masked_targets = np.where(mask, targets, np.nan)  # 确保 targets 是 DataArray
    return masked_preds, masked_targets

def calculate_rmse(predictions: xr.Dataset, targets: xr.Dataset, mask: xr.DataArray, variable: str) -> float:
    pred_values = jax.device_get(predictions[variable].data)
    mask = jax.device_get(mask[variable].data)
    target_values = targets[variable].values
    pred_values = np.transpose(pred_values, (1, 0, 2, 3))
    # mask = np.transpose(mask, (1, 0, 2, 3))
    pred_values, target_values = apply_mask(pred_values, target_values, mask)
    
    # 如果包含 `level` 维度，对 `level` 维度取均值
    if 'level' in predictions[variable].dims:
        pred_values = np.nanmean(pred_values, axis=-2)  # 对 `level` 维度取均值
        target_values = np.nanmean(target_values, axis=-2)
        mask_values = np.nanmean(mask_values, axis=-2)

    mse = np.nanmean((pred_values - target_values) ** 2)
    rmse = np.sqrt(mse)
    return rmse

def calculate_mse(predictions: xr.Dataset, targets: xr.Dataset, mask: xr.DataArray, variable: str) -> float:
    pred_values = jax.device_get(predictions[variable].data)
    mask = jax.device_get(mask[variable].data)
    target_values = targets[variable].values
    pred_values = np.transpose(pred_values, (1, 0, 2, 3))
    # mask = np.transpose(mask, (1, 0, 2, 3))
    pred_values, target_values = apply_mask(pred_values, target_values, mask)
    
    # 如果包含 `level` 维度，对 `level` 维度取均值
    if 'level' in predictions[variable].dims:
        pred_values = np.nanmean(pred_values, axis=-2)  # 对 `level` 维度取均值
        target_values = np.nanmean(target_values, axis=-2)
        mask_values = np.nanmean(mask_values, axis=-2)

    mse = np.nanmean((pred_values - target_values) ** 2)
    return mse

def calculate_mae(predictions: xr.Dataset, targets: xr.Dataset, mask: xr.DataArray, variable: str) -> float:
    pred_values = jax.device_get(predictions[variable].data)
    mask = jax.device_get(mask[variable].data)
    target_values = targets[variable].values
    pred_values = np.transpose(pred_values, (1, 0, 2, 3))
    # mask = np.transpose(mask, (1, 0, 2, 3))
    pred_values, target_values = apply_mask(pred_values, target_values, mask)
    
    # 如果包含 `level` 维度，对 `level` 维度取均值
    if 'level' in predictions[variable].dims:
        pred_values = np.nanmean(pred_values, axis=-2)  # 对 `level` 维度取均值
        target_values = np.nanmean(target_values, axis=-2)
        mask_values = np.nanmean(mask_values, axis=-2)

    mae = np.nanmean(np.abs(pred_values - target_values))
    return mae


# def apply_mask(predictions, targets, mask):
#     """应用掩码，只保留有效值。"""
#     masked_preds = np.where(mask, predictions, np.nan)
#     masked_targets = np.where(mask, targets, np.nan)
#     return masked_preds, masked_targets

def calculate_metrics_per_level(predictions: xr.Dataset, 
                                targets: xr.Dataset, 
                                mask: xr.DataArray, 
                                variable: str, 
                                metric: str) -> np.ndarray:
    """
    针对每个level计算指定变量的MAE、RMSE或MSE。

    参数：
        predictions: 预测数据集
        targets: 实际数据集
        mask: 掩码
        variable: 目标变量名
        metric: "mae", "rmse" 或 "mse"

    返回：
        一个包含每个level计算结果的数组
    """
    pred_values = jax.device_get(predictions[variable].data)
    mask = jax.device_get(mask[variable].data)
    target_values = targets[variable].values

    # 调整维度顺序以对齐
    pred_values = np.transpose(pred_values, (1, 0, 2, 3, 4))  # 调整后(time, batch, level, lat, lon)

    # 应用掩码
    pred_values, target_values = apply_mask(pred_values, target_values, mask)

    # 针对每个level计算指标
    results = []
    num_levels = pred_values.shape[2]
    for level in range(num_levels):
        pred_level = pred_values[..., level]
        target_level = target_values[..., level]

        if metric == "mae":
            result = np.nanmean(np.abs(pred_level - target_level))
        elif metric == "rmse":
            mse = np.nanmean((pred_level - target_level) ** 2)
            result = np.sqrt(mse)
        elif metric == "mse":
            result = np.nanmean((pred_level - target_level) ** 2)
        else:
            raise ValueError("Unsupported metric: choose from 'mae', 'rmse', or 'mse'.")

        results.append(result)

    return np.array(results)

In [21]:
import jax
import jax.numpy as jnp
from jax import random
import numpy as np

total_rmse_siconc = 0
total_rmse_sithick = 0
total_mse_siconc = 0
total_mse_sithick = 0
total_mae_siconc = 0
total_mae_sithick = 0

total_mae_so = [0] * 2
total_rmse_so = [0] * 2
total_mse_so = [0] * 2

total_mae_thetao = [0] * 2
total_rmse_thetao = [0] * 2
total_mse_thetao = [0] * 2

# 文件路径
dir_path_data = "/root/autodl-tmp/"
save_path = "/root/data/stats/"

# 获取所有 NetCDF 文件路径
input_files = glob.glob(f"{dir_path_data}/eval_batch.nc")
# 逐个文件读取并训练
for file_path in input_files:
    print(f"正在处理文件：{file_path}")
    # 读取当前文件的 dataset
    eval_batch = xr.open_dataset(file_path, chunks={'batch': 1})
    
    batch_size = eval_batch.dims['batch']  # 根据你的数据调整
    print("batchsize:", batch_size)
    
    eval_batch["land_sea_mask"] = (
    (~np.isnan(eval_batch["so"].isel(time=0, level=0)))
    .astype(np.float32)
    )
    eval_batch = eval_batch.fillna(0)
    eval_batch = eval_batch.astype(np.float32)
    
    for start_idx in range(batch_size):
        
        
        test_batch = xr.Dataset({var: eval_batch[var].isel(batch=slice(start_idx, start_idx + 1)) for var in eval_batch.data_vars})
        test_batch.load()
        # print(test_batch.dims.mapping)
        
        eval_inputs, eval_targets, eval_forcings = data_utils.extract_inputs_targets_forcings(
            test_batch,
            target_lead_times=slice("24h", f"{eval_steps.value * 24}h"),
            **dataclasses.asdict(task_config))
        
        predictions = run_forward_jitted(
        rng=random.PRNGKey(0),
        inputs=eval_inputs,
        targets_template=eval_targets * jnp.nan,
        forcings=eval_forcings
        )
        
            # 创建掩码，数值为0的部分为False，其他部分为True
        eval_targetmask = eval_targets != 0  # 将数值为0的部分设为False，其他部分设为True
        # 将掩码转换为float32类型（可选，具体看你后续需要使用的格式）
        eval_targetmask = eval_targetmask.astype(np.float32)
        eval_targetmask
            
        rmse_siconc = calculate_rmse(predictions, eval_targets, eval_targetmask, 'siconc')
        rmse_sithick = calculate_rmse(predictions, eval_targets, eval_targetmask, 'sithick')
        
        mse_siconc = calculate_mse(predictions, eval_targets, eval_targetmask, 'siconc')
        mse_sithick = calculate_mse(predictions, eval_targets, eval_targetmask, 'sithick')
        
        mae_siconc = calculate_mae(predictions, eval_targets, eval_targetmask, 'siconc')
        mae_sithick = calculate_mae(predictions, eval_targets, eval_targetmask, 'sithick')
        
        # 示例用法
        mae_so = calculate_metrics_per_level(predictions, eval_targets, eval_targetmask, 'so', 'mae')
        rmse_so = calculate_metrics_per_level(predictions, eval_targets, eval_targetmask, 'so', 'rmse')
        mse_so = calculate_metrics_per_level(predictions, eval_targets, eval_targetmask, 'so', 'mse')
        
        mae_thetao = calculate_metrics_per_level(predictions, eval_targets, eval_targetmask, 'thetao', 'mae')
        rmse_thetao = calculate_metrics_per_level(predictions, eval_targets, eval_targetmask, 'thetao', 'rmse')
        mse_thetao = calculate_metrics_per_level(predictions, eval_targets, eval_targetmask, 'thetao', 'mse')
    
        # 累加当前批次的损失
        total_rmse_siconc += rmse_siconc
        total_rmse_sithick += rmse_sithick
        total_mse_siconc += mse_siconc
        total_mse_sithick += mse_sithick
        total_mae_siconc += mae_siconc
        total_mae_sithick += mae_sithick
        
        for i, (mae, rmse, mse) in enumerate(zip(mae_so, rmse_so, mse_so)):
            total_mae_so[i] += mae
            total_rmse_so[i] += rmse
            total_mse_so[i] += mse
    
        for i, (mae, rmse, mse) in enumerate(zip(mae_thetao, rmse_thetao, mse_thetao)):
            total_mae_thetao[i] += mse
            total_rmse_thetao[i] += mae
            total_mse_thetao[i] += mae


正在处理文件：/root/autodl-tmp//eval_batch.nc
batchsize: 1


[Patch-Grid] num_grid_nodes = 8817120, num_patch_nodes = 138240, patch_size_lat = 8, patch_size_lon = 8
Patch-Grid2Mesh edges: 216800
[Grid2Mesh] num_grid_nodes = 138240, num_edges = 216800, neighbors per node: min=1, mean=1.57, max=3
Mesh2Grid Edges: 17634240
stacked_array.sizes: Frozen({'batch': 1, 'lat': 2041, 'lon': 4320, 'channels': 14})
template_dataset.sizes: Frozen({'batch': 1, 'time': 1, 'lat': 2041, 'lon': 4320, 'level': 3})
stacked_array: <xarray.Variable (batch: 1, lat: 2041, lon: 4320, channels: 14)>
xarray_jax.JaxArrayWrapper(Traced<ShapedArray(float32[1,2041,4320,14])>with<DynamicJaxprTrace(level=2/0)>)
template_dataset: <xarray.Dataset>
Dimensions:  (batch: 1, time: 1, lat: 2041, lon: 4320, level: 3)
Coordinates:
  * lon      (lon) float32 -180.0 -179.9 -179.8 -179.8 ... 179.8 179.8 179.9
  * lat      (lat) float32 -80.0 -79.92 -79.83 -79.75 ... 89.75 89.83 89.92 90.0
  * time     (time) timedelta64[ns] 1 days
  * level    (level) float32 0.494 2.646 5.078
Dimensions wi

2026-04-14 18:12:04.114440: E external/xla/xla/service/slow_operation_alarm.cc:65] Constant folding an instruction is taking > 1s:

  %scatter.33 = f32[138240]{0} scatter(f32[138240]{0} %broadcast.388, s32[8817120,1]{1,0} %constant.153, f32[8817120,1]{1,0} %broadcast.389), update_window_dims={1}, inserted_window_dims={}, scatter_dims_to_operand_dims={0}, index_vector_dim=1, to_apply=%region_2.224, metadata={op_name="jit(<unnamed wrapped function>)/jit(main)/while/body/patch_pool_mlp/scatter-add[update_consts=() dimension_numbers=ScatterDimensionNumbers(update_window_dims=(), inserted_window_dims=(0,), scatter_dims_to_operand_dims=(0,)) indices_are_sorted=False unique_indices=False mode=GatherScatterMode.FILL_OR_DROP]" source_file="/root/code/code/GraphCast-from-Ground-Zero/graphcast/graphcast.py" source_line=362}

This isn't necessarily a bug; constant-folding is inherently a trade-off between compilation time and speed at runtime. XLA has some guards that attempt to keep constant fold

IndexError: list index out of range

In [ ]:
avg_mae_so = [0] * 2
avg_rmse_so = [0] * 2
avg_mse_so = [0] * 2
avg_mae_thetao = [0] * 2
avg_rmse_thetao = [0] * 2
avg_mse_thetao = [0] * 2

batch_size = 30

# 计算平均值
avg_rmse_siconc = total_rmse_siconc / batch_size
avg_rmse_sithick = total_rmse_sithick / batch_size
avg_mse_siconc = total_mse_siconc / batch_size
avg_mse_sithick = total_mse_sithick / batch_size
avg_mae_siconc = total_mae_siconc / batch_size
avg_mae_sithick = total_mae_sithick / batch_size

for i, (mae, rmse, mse) in enumerate(zip(total_mae_so, total_rmse_so, total_mse_so)): 
    avg_mae_so[i] = total_mae_so[i] / batch_size
    avg_rmse_so[i] = total_rmse_so[i] / batch_size
    avg_mse_so[i] = total_mse_so[i] / batch_size

for i, (mae, rmse, mse) in enumerate(zip(total_mae_thetao, total_rmse_thetao, total_mse_thetao)):
    avg_mae_thetao[i] = total_mae_thetao[i] / batch_size
    avg_rmse_thetao[i] = total_rmse_thetao[i] / batch_size
    avg_mse_thetao[i] = total_mse_thetao[i] / batch_size


# 打印平均值
print(f"Average RMSE (siconc): {avg_rmse_siconc}")
print(f"Average RMSE (sithick): {avg_rmse_sithick}")
print(f"Average MSE (siconc): {avg_mse_siconc}")
print(f"Average MSE (sithick): {avg_mse_sithick}")
print(f"Average MAE (siconc): {avg_mae_siconc}")
print(f"Average MAE (sithick): {avg_mae_sithick}")

# 打印结果
for i, (mae, rmse, mse) in enumerate(zip(avg_mae_so, avg_rmse_so, avg_mse_so)):
    print(f"so 的 level {i} -> MAE: {mae}, RMSE: {rmse}, MSE: {mse}")

for i, (mae, rmse, mse) in enumerate(zip(avg_mae_thetao, avg_rmse_thetao, avg_mse_thetao)):
    print(f"thetao 的 level {i} -> MAE: {mae}, RMSE: {rmse}, MSE: {mse}")

In [ ]:
# example_batch = 

In [ ]:
# @title Choose data to plot

plot_example_variable = widgets.Dropdown(
    options=example_batch.data_vars.keys(),
    value="2m_temperature",
    description="Variable")
plot_example_level = widgets.Dropdown(
    options=example_batch.coords["level"].values,
    value=500,
    description="Level")
plot_example_robust = widgets.Checkbox(value=True, description="Robust")
plot_example_max_steps = widgets.IntSlider(
    min=1, max=example_batch.dims["time"], value=example_batch.dims["time"],
    description="Max steps")

widgets.VBox([
    plot_example_variable,
    plot_example_level,
    plot_example_robust,
    plot_example_max_steps,
    widgets.Label(value="Run the next cell to plot the data. Rerunning this cell clears your selection.")
])

In [ ]:
# @title Plot example data

plot_size = 7

data = {
    " ": scale(select(example_batch, plot_example_variable.value, plot_example_level.value, plot_example_max_steps.value),
              robust=plot_example_robust.value),
}
fig_title = plot_example_variable.value
if "level" in example_batch[plot_example_variable.value].coords:
  fig_title += f" at {plot_example_level.value} hPa"

plot_data(data, fig_title, plot_size, plot_example_robust.value)


In [ ]:
# import jax
# import jax.numpy as jnp
# from jax import random
# import numpy as np

# predictions = run_forward_jitted(
#     rng=random.PRNGKey(0),
#     inputs=eval_inputs,
#     targets_template=eval_targets * jnp.nan,
#     forcings=eval_forcings
# )

# # 输出预测结果
# predictions

In [ ]:
# import xarray as xr
# import numpy as np
# import jax

# def apply_mask(predictions, targets, mask):
#     """应用掩码，只保留有效值。"""
#     # 将 xarray.Dataset 中的 DataArray 转换为 numpy 数组
#     masked_preds = np.where(mask, predictions, np.nan)  # 确保 predictions 是 DataArray
#     masked_targets = np.where(mask, targets, np.nan)  # 确保 targets 是 DataArray
#     return masked_preds, masked_targets

# def calculate_rmse(predictions: xr.Dataset, targets: xr.Dataset, mask: xr.DataArray, variable: str) -> float:
#     pred_values = jax.device_get(predictions[variable].data)
#     mask = jax.device_get(mask[variable].data)
#     target_values = targets[variable].values
#     pred_values = np.transpose(pred_values, (1, 0, 2, 3))
#     # mask = np.transpose(mask, (1, 0, 2, 3))
#     pred_values, target_values = apply_mask(pred_values, target_values, mask)
    
#     # 如果包含 `level` 维度，对 `level` 维度取均值
#     if 'level' in predictions[variable].dims:
#         pred_values = np.nanmean(pred_values, axis=-2)  # 对 `level` 维度取均值
#         target_values = np.nanmean(target_values, axis=-2)
#         mask_values = np.nanmean(mask_values, axis=-2)

#     mse = np.nanmean((pred_values - target_values) ** 2)
#     rmse = np.sqrt(mse)
#     return rmse

# def calculate_mse(predictions: xr.Dataset, targets: xr.Dataset, mask: xr.DataArray, variable: str) -> float:
#     pred_values = jax.device_get(predictions[variable].data)
#     mask = jax.device_get(mask[variable].data)
#     target_values = targets[variable].values
#     pred_values = np.transpose(pred_values, (1, 0, 2, 3))
#     # mask = np.transpose(mask, (1, 0, 2, 3))
#     pred_values, target_values = apply_mask(pred_values, target_values, mask)
    
#     # 如果包含 `level` 维度，对 `level` 维度取均值
#     if 'level' in predictions[variable].dims:
#         pred_values = np.nanmean(pred_values, axis=-2)  # 对 `level` 维度取均值
#         target_values = np.nanmean(target_values, axis=-2)
#         mask_values = np.nanmean(mask_values, axis=-2)

#     mse = np.nanmean((pred_values - target_values) ** 2)
#     return mse

# def calculate_mae(predictions: xr.Dataset, targets: xr.Dataset, mask: xr.DataArray, variable: str) -> float:
#     pred_values = jax.device_get(predictions[variable].data)
#     mask = jax.device_get(mask[variable].data)
#     target_values = targets[variable].values
#     pred_values = np.transpose(pred_values, (1, 0, 2, 3))
#     # mask = np.transpose(mask, (1, 0, 2, 3))
#     pred_values, target_values = apply_mask(pred_values, target_values, mask)
    
#     # 如果包含 `level` 维度，对 `level` 维度取均值
#     if 'level' in predictions[variable].dims:
#         pred_values = np.nanmean(pred_values, axis=-2)  # 对 `level` 维度取均值
#         target_values = np.nanmean(target_values, axis=-2)
#         mask_values = np.nanmean(mask_values, axis=-2)

#     mae = np.nanmean(np.abs(pred_values - target_values))
#     return mae

# # 示例用法
# rmse_siconc = calculate_rmse(predictions, eval_targets, eval_targetmask, 'siconc')
# rmse_sithick = calculate_rmse(predictions, eval_targets, eval_targetmask, 'sithick')

# mse_siconc = calculate_mse(predictions, eval_targets, eval_targetmask, 'siconc')
# mse_sithick = calculate_mse(predictions, eval_targets, eval_targetmask, 'sithick')

# mae_siconc = calculate_mae(predictions, eval_targets, eval_targetmask, 'siconc')
# mae_sithick = calculate_mae(predictions, eval_targets, eval_targetmask, 'sithick')

# print(f"siconc 的 RMSE: {rmse_siconc}")
# print(f"sithick 的 RMSE: {rmse_sithick}")
# print(f"siconc 的 MSE: {mse_siconc}")
# print(f"sithick 的 MSE: {mse_sithick}")
# print(f"siconc 的 MAE: {mae_siconc}")
# print(f"sithick 的 MAE: {mae_sithick}")

In [ ]:
# import xarray as xr 
# import numpy as np
# import jax

# def apply_mask(predictions, targets, mask):
#     """应用掩码，只保留有效值。"""
#     masked_preds = np.where(mask, predictions, np.nan)
#     masked_targets = np.where(mask, targets, np.nan)
#     return masked_preds, masked_targets

# def calculate_metrics_per_level(predictions: xr.Dataset, 
#                                 targets: xr.Dataset, 
#                                 mask: xr.DataArray, 
#                                 variable: str, 
#                                 metric: str) -> np.ndarray:
#     """
#     针对每个level计算指定变量的MAE、RMSE或MSE。

#     参数：
#         predictions: 预测数据集
#         targets: 实际数据集
#         mask: 掩码
#         variable: 目标变量名
#         metric: "mae", "rmse" 或 "mse"

#     返回：
#         一个包含每个level计算结果的数组
#     """
#     pred_values = jax.device_get(predictions[variable].data)
#     mask = jax.device_get(mask[variable].data)
#     target_values = targets[variable].values

#     # 调整维度顺序以对齐
#     pred_values = np.transpose(pred_values, (1, 0, 2, 3, 4))  # 调整后(time, batch, level, lat, lon)

#     # 应用掩码
#     pred_values, target_values = apply_mask(pred_values, target_values, mask)

#     # 针对每个level计算指标
#     results = []
#     num_levels = pred_values.shape[2]
    
#     for level in range(num_levels):
#         pred_level = pred_values[..., level]
#         target_level = target_values[..., level]

#         if metric == "mae":
#             result = np.nanmean(np.abs(pred_level - target_level))
#         elif metric == "rmse":
#             mse = np.nanmean((pred_level - target_level) ** 2)
#             result = np.sqrt(mse)
#         elif metric == "mse":
#             result = np.nanmean((pred_level - target_level) ** 2)
#         else:
#             raise ValueError("Unsupported metric: choose from 'mae', 'rmse', or 'mse'.")

#         results.append(result)

#     return np.array(results)

# # 示例用法
# mae_so = calculate_metrics_per_level(predictions, eval_targets, eval_targetmask, 'so', 'mae')
# rmse_so = calculate_metrics_per_level(predictions, eval_targets, eval_targetmask, 'so', 'rmse')
# mse_so = calculate_metrics_per_level(predictions, eval_targets, eval_targetmask, 'so', 'mse')

# mae_siconc = calculate_metrics_per_level(predictions, eval_targets, eval_targetmask, 'thetao', 'mae')
# rmse_siconc = calculate_metrics_per_level(predictions, eval_targets, eval_targetmask, 'thetao', 'rmse')
# mse_siconc = calculate_metrics_per_level(predictions, eval_targets, eval_targetmask, 'thetao', 'mse')

# # 打印结果
# for i, (mae, rmse, mse) in enumerate(zip(mae_so, rmse_so, mse_so)):
#     print(f"so 的 level {i} -> MAE: {mae}, RMSE: {rmse}, MSE: {mse}")

# for i, (mae, rmse, mse) in enumerate(zip(mae_siconc, rmse_siconc, mse_siconc)):
#     print(f"thetao 的 level {i} -> MAE: {mae}, RMSE: {rmse}, MSE: {mse}")

    

In [ ]:
# @title 选择要绘制的预测结果

# 假设predictions是你的xarray Dataset对象
print(predictions.coords.keys())  # 打印所有坐标键

plot_pred_variable = widgets.Dropdown(
    options=predictions.data_vars.keys(),
    value="siconc",
    description="变量")
plot_pred_level = widgets.Dropdown(
    options=eval_batch.coords["level"].values,
    value=np.float32(0.494025),
    description="深度")
plot_pred_robust = widgets.Checkbox(value=True, description="鲁棒性")
plot_pred_max_steps = widgets.IntSlider(
    min=1,
    max=predictions.dims["time"],
    value=predictions.dims["time"],
    description="最大步")

widgets.VBox([
    plot_pred_variable,
    plot_pred_level,
    plot_pred_robust,
    plot_pred_max_steps,
    widgets.Label(value="运行下一个单元格，绘制预测结果。重新运行该单元格将清除您的选择。")
])

In [ ]:
# 设置绘图标题
# @title 使用预测数据绘图
# 将predictions逐变量提取数据，并重新组织成一个xarray.Dataset
import xarray as xr
import jax
# 设置绘图大小
plot_size = 5

# 使用jax.device_get提取数据
# variables = {var: (predictions[var].dims, jax.device_get(predictions[var].data)) for var in predictions.data_vars}

# 重新创建xarray.Dataset
# prediction = xr.Dataset(variables, coords=predictions.coords)

# # 1. 提取mask中的四个变量并重新组织成一个数据集
# mask_vars = ['siconc', 'sithick', 'so', 'thetao']
# mask_data = {var: (eval_targetmask[var].dims, jax.device_get(eval_targetmask[var].data)) for var in mask_vars}

# # 创建一个新的xarray.Dataset用于存储这四个变量的掩码
# mask_dataset = xr.Dataset(mask_data, coords=mask.coords)
processed_variables = {}

for variable in predictions.data_vars:
    pred_values = jax.device_get(predictions[variable].data)
    
#     if pred_values.ndim == 4:
#         # 如果是四维数组，使用四维的轴顺序
#         pred_values = np.transpose(pred_values, (1, 0, 2, 3))
#     elif pred_values.ndim == 5:
#         # 如果是五维数组，使用五维的轴顺序
#         pred_values = np.transpose(pred_values, (1, 0, 2, 3, 4))
    
    mask_values = jax.device_get(eval_targetmask[variable].data)
    
    if mask_values.ndim == 4:
        # 如果是四维数组，使用四维的轴顺序
        mask_values = np.transpose(mask_values, (1, 0, 2, 3))
    elif mask_values.ndim == 5:
        # 如果是五维数组，使用五维的轴顺序
        mask_values = np.transpose(mask_values, (1, 0, 2, 3, 4))
    
    # 打印每个变量的名称及其维度信息
    print(f"Variable: {variable}")
    print(f"pred_values shape: {pred_values.shape}, dtype: {pred_values.dtype}")
    print(f"mask_values shape: {mask_values.shape}, dtype: {mask_values.dtype}")
    
    masked_values = np.where(mask_values, pred_values, np.nan)
    
     # 打印 masked_values 的维度和类型信息
    print(f"masked_values shape: {masked_values.shape}, dtype: {masked_values.dtype}")
    
     # 将处理后的变量存入字典中，每个变量都是一个 xarray.DataArray
    processed_variables[variable] = xr.DataArray(
        masked_values, dims=predictions[variable].dims, coords=predictions[variable].coords
    )

    # 将所有处理后的变量重新组合成一个新的 xarray.Dataset
prediction = xr.Dataset(processed_variables, coords=predictions.coords)
prediction
# 计算最大步数，用于绘图
plot_max_steps = min(prediction.dims["time"], plot_pred_max_steps.value)

# 准备绘图数据
data = {
    # 缩放目标数据 
    # "Inputs": scale(select(eval_inputs, plot_pred_variable.value,  plot_max_steps), robust=plot_pred_robust.value),
    "Targets": scale(select(eval_targets, plot_pred_variable.value,  plot_max_steps), robust=plot_pred_robust.value),
    # 缩放预测数据
    "Predictions": scale(select(prediction, plot_pred_variable.value, plot_max_steps), robust=plot_pred_robust.value),
    # 计算并缩放目标与预测的差异
    "Diff": scale((select(eval_targets, plot_pred_variable.value, plot_max_steps) -
                        select(prediction, plot_pred_variable.value, plot_max_steps)),
                       robust=plot_pred_robust.value, center=0),
}

# "Inputs": scale(select(eval_inputs, plot_pred_variable.value,  plot_max_steps), robust=plot_pred_robust.value),

# 设置图表标题
fig_title = plot_pred_variable.value
# 如果预测数据中包含“level”坐标，则在标题中添加相应的气压层级
if "level" in predictions[plot_pred_variable.value].coords:
  fig_title += f" at {plot_pred_level.value} hPa"

# 调用plot_data函数进行绘图
plot_data(data, fig_title, plot_size, plot_pred_robust.value)


In [ ]:
# # 保存为 NetCDF 文件
# output_file_path = "/root/autodl-tmp/predictions.nc"
# prediction.to_netcdf(output_file_path)

# print(f"Predictions 已成功保存至 {output_file_path}")